## Despliegue en Render 

El proyecto tiene la siguiente estructura:

mi-api-flask/         
├── app.py  
├── model.pkl  
├── requirements.txt  
├── README.md  

Los pasos seguidos para el despliegue fueron los siguientes:  

#### 1- Subir a GitHub la API y el modelo al repositorio de GitHub:  

    **https://github.com/arielaastorga/ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-**    


Render crea el servicio conectando directamente un repositorio y desplegándolo como Web Service.  

#### 2- Crear servicio  

En Render, dentro al dashboard, se crea un servicio nuevo desde New > Web Service.  
Region: está ubicada en Oregon (US)  
Se conecta a la cuenta de GitHub y selecciona el repositorio de la API.  
Durante la creación del servicio, en la documentación de Render se indica usar estos valores base:   

- Branch:	main    
- Root Directory: Proyecto_prod   **Porque el proyecto está ubicado en una subcarpeta 
- Build Command: pip install -r requirements.txt  
- lenguaje Python 3  
- Start command gunicorn app:app.

#### 3- Pulsar Create Web Service   

Render descarga el repo, instala dependencias y arranca la API. Al terminar, dará una URL pública onrender.com.
En este caso:    

**https://online-ds-bridge-proyects-arielaastorga.onrender.com/**

#### 4- Probar despliegue

Cuando el deploy termina, abrir la URL pública y probar las rutas. 

Página de inicio  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/   

Health  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/health  

PATH  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/neighbourhood/Embajadores  

Querys:  
Piso entero en Sol  

  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/predict-query?accommodates=4&bathrooms=1&bedrooms=2&beds=2&minimum_nights=2&number_of_reviews=50&review_scores_rating=4&availability_365=180&neighbourhood_cleansed=Sol&room_type=Entire%20home/apt  


{"predicted_price":183.29}

Habitación privada en Embajadores  

https://online-ds-bridge-proyects-arielaastorga.onrender.com/predict-query?accommodates=2&bathrooms=1&bedrooms=1&beds=1&minimum_nights=3&number_of_reviews=120&review_scores_rating=4&availability_365=90&neighbourhood_cleansed=Embajadores&room_type=Private%20room  


{"predicted_price":87.64}

## Despliegue en AWS 

🏗️ ARQUITECTURA FINAL
INTERNET  
    ↓  
Puerto 80  →  NGINX       (recibe peticiones de internet)  
    ↓  
Puerto 8000 →  GUNICORN   (gestiona workers Python)  
    ↓  
Puerto 5000 →  FLASK      (app.py: carga el modelo y devuelve predicciones)  

##### Parte 1 — CREAR LA INSTANCIA EN AWS EC2  

**1.1 Configuración de la instancia**  

Ir a AWS EC2 → "Lanzar la instancia"    
    Nombre: ml-project-airbnb    
    AMI: Ubuntu Server 24.04 LTS (64 bits x86)    
    Tipo de instancia: t3.small (2 vCPU / 2 GB RAM)  

Problema 1:  
Aquí tuvimos que crear una instancia de mayor tamaño para poder instalar Python 3.12. Más adelante se detalla.

Par de claves: se utilizó la clave existente creada para el proyecto en clase: .pem    

Red:  
IP pública: Habilitar  
Firewall: Permitir SSH, HTTP, HTTPS  

Almacenamiento: 20 GB (tipo gp3)  
Clic en "Lanzar instancia"  

**1.2 Datos de conexión de la instancia**  

DNS público: ec2-51-20-77-195.eu-north-1.compute.amazonaws.com  
Región: Europa (Estocolmo) eu-north-1  
Usuario: ubuntu  



##### Parte 2 — CONECTARSE CON PUTTY

**2.1 Convertir .pem a .ppk con PuTTYgen**  

Abrir PuTTYgen  
Clic en "Load" → cambiar filtro a "All Files (.)"
Seleccionar el archivo .pem
Clic en "Save private key" → guardar como .ppk

**2.2 Configurar PuTTY**

Host Name: ubuntu@ec2-13-48-135-255.eu-north-1.compute.amazonaws.com
Port: 22    
Connection type: SSH    
Ir a Connection → SSH → Auth → Credentials    
Cargar el archivo .ppk    
Guardar sesión → clic en "Open"    



##### PARTE 3 — CONFIGURAR EL SERVIDOR  

**3.1 Actualizar el sistema**  
bashsudo apt update && sudo apt upgrade -y  

Tuvimos que crear un Swap porque dejaba elegir una instancia small o xlarge. Una opción que fue la elegida fue usar la instancia small, y 
hacer un swap para usar la memoria del disco para hacer la instación de Python12.
La opción de usar la instancia Xlarga no era tan buena porque se iban a consumir los créditos muy rápido.


**3.2 Crear Swap (memoria virtual extra)**  

El swap permite usar el disco como RAM adicional durante instalaciones pesadas.  
bashsudo fallocate -l 2G /swapfile    
sudo chmod 600 /swapfile    
sudo mkswap /swapfile  
sudo swapon /swapfile  
echo '/swapfile none swap sw 0 0' | sudo tee -a /etc/fstab  
Verificar que está activo:  
bashfree -h  
Swap: 2.0Gi ✅  

##### PARTE 4 — INSTALAR PYTHON 3.12  

Nos encontramos aquí con un problema de compatibilidad: Ubuntu 26.04 trae Python 3.14 por defecto, pero NumPy 1.26.4 solo es compatible  
hasta Python 3.12. Por eso se compila Python 3.12 desde código fuente.

bash# Instalar dependencias para compilar  
sudo apt install -y build-essential zlib1g-dev libncurses5-dev libgdbm-dev \  
libnss3-dev libssl-dev libreadline-dev libffi-dev libsqlite3-dev wget  

*Descargar Python 3.12*  
wget https://www.python.org/ftp/python/3.12.7/Python-3.12.7.tgz  

*Descomprimir* 
tar -xf Python-3.12.7.tgz  
cd Python-3.12.7  

*Compilar*  
./configure --enable-optimizations
make -j$(nproc)
sudo make altinstall

*Verificar*
python3.12 --version
Python 3.12.7 ✅

Volver al home
cd ~


##### PARTE 5 — ESTRUCTURA DEL PROYECTO  

/home/ubuntu/  
└── prod/  
    ├── ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-/   ← repo GitHub  
    │   └── Proyecto_prod/  
    │       ├── app.py  
    │       └── model/  
    │           └── model.pkl   
    └── endpoint/                                      ← entorno virtual  
        ├── app.py   → enlace simbólico a app.py  
        ├── model    → enlace simbólico a carpeta model  
        ├── logs/  
        ├── gunicorn.conf.py  
        └── venv/  

**5.1 Crear la estructura**  
bash# Crear carpeta principal  
mkdir prod  
cd prod  

**Clonar repositorio de GitHub**  
git clone https://github.com/arielaastorga/ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-  

**Crear carpeta endpoint**  
mkdir endpoint  
cd endpoint  

**5.2 Crear entorno virtual e instalar librerías**  
bash# Crear entorno virtual con Python 3.12  
python3.12 -m venv venv  

**Activar entorno virtual**  
source venv/bin/activate  
Aparece (venv) al inicio de la línea ✅  

**Instalar librerías compatibles con Python 3.12**  
pip install flask  
pip install numpy==1.26.4  
pip install pandas  
pip install scikit-learn  
pip install joblib  
pip install gunicorn  

**5.3 Crear enlaces simbólicos (accesos directos al repo)**  
ln -s "../ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-/Proyecto_prod/app.py"  
ln -s "../ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-/Proyecto_prod/model"  

**5.4 Añadir bloque de arranque al final de app.py**  
bashnano app.py  

Problema 2: Aquí tuvimos que modificar el archivo para indicarle como inicializar App
(lo cual no era necesario en Render)

Ir al final del archivo con CTRL+END y añadir:  

pythonif __name__ == "__main__":  
    app.run(host="0.0.0.0", port=5000)  
Guardar: CTRL+X → Y → Enter  

**5.5 Probar que Flask funciona**  
bash# Terminal 1: arrancar la app  
python app.py  

**Terminal 2: probar endpoints**
Abrimos un nuevo terminal y probamos que funciona  
curl 127.0.0.1:5000/health

#### PARTE 6 — CONFIGURAR GUNICORN

Gunicorn es el servidor WSGI que gestiona múltiples peticiones simultáneas.  

**6.1 Crear carpeta de logs**  
bashmkdir logs

**6.2 Crear archivo de configuración**    
bashnano gunicorn.conf.py  
pythonbind="0.0.0.0:8000"  
workers=1  
timeout=120  
accesslog="/home/ubuntu/prod/endpoint/logs/gunicorn.access.log"  
errorlog="/home/ubuntu/prod/endpoint/logs/gunicorn.error.log"  
capture_output=True  
loglevel="info"  


Guardar: CTRL+X → Y → Enter  

**6.3 Probar Gunicorn**  
bash/home/ubuntu/prod/endpoint/venv/bin/gunicorn \  
  -c /home/ubuntu/prod/endpoint/gunicorn.conf.py app:app  

En otra terminal:  
curl 127.0.0.1:8000/health  
{"status": "ok", "service": "running"} ✅    

##### PARTE 7 — GUNICORN COMO SERVICIO PERMANENTE  

Esto hace que la API arranque automáticamente aunque se reinicie el servidor.  
bashsudo nano /etc/systemd/system/mlairbnb.service  
ini[Unit]  
Description=ML Airbnb Madrid API  
After=network.target  

[Service]  
User=ubuntu  
WorkingDirectory=/home/ubuntu/prod/endpoint  
Environment="PATH=/home/ubuntu/prod/endpoint/venv/bin"  
ExecStart=/home/ubuntu/prod/endpoint/venv/bin/gunicorn -c /home/ubuntu/prod/endpoint/gunicorn.conf.py app:app  
Restart=always  

[Install]  
WantedBy=multi-user.target  
Guardar: CTRL+X → Y → Enter  
bashsudo systemctl daemon-reload  
sudo systemctl start mlairbnb.service  
sudo systemctl enable mlairbnb.service  
sudo systemctl status mlairbnb.service  
Active: active (running) ✅  

#### PARTE 8 — CONFIGURAR NGINX  
Nginx es el servidor web que expone la API a internet a través del puerto 80.  

**8.1 Instalar Nginx**  

bashsudo apt-get install nginx -y  

**8.2 Crear configuración**  

bashsudo nano /etc/nginx/sites-available/mlairbnb  
nginxserver {  
    listen 80;  
    server_name 51.20.77.195;  
    location / {  
        proxy_pass http://127.0.0.1:8000;  
    }  
}  
Guardar: CTRL+X → Y → Enter  

**8.3 Activar la configuración**  

bashsudo ln -s /etc/nginx/sites-available/mlairbnb /etc/nginx/sites-enabled/  
sudo systemctl start nginx  
sudo systemctl enable nginx  
sudo systemctl status nginx  
Active: active (running) ✅  

**8.4 Abrir puerto 80 en AWS**  

AWS EC2 → Grupos de seguridad  
Seleccionar el grupo de la instancia  
Editar reglas de entrada → Añadir regla  

Tipo: HTTP | Puerto: 80 | Origen: 0.0.0.0/0
Guardar

##### PARTE 9 — PROBAR LA API DESDE INTERNET

bash# Health check  
curl http://IP_publica/health  



##### PARA VOLVER A UTILIZAR EL SERVICIO
Reiniciar el servicio tras cambios en app.py  
sudo systemctl restart mlairbnb.service  

- Ver estado del servicio  
sudo systemctl status mlairbnb.service  

- Actualizar código desde GitHub  
cd /home/ubuntu/prod/ONLINE_DS_BRIDGE_-Proyects_ArielaAstorga-  
git pull   
sudo systemctl restart mlairbnb.service  

- Detener la instancia para no gastar créditos
Hacerlo desde el panel AWS EC2 → Instancias → Estado → Detener

Pasos:  

PASO 1 — Verificar que el servicio está corriendo  
sudo systemctl status mlairbnb.service  

PASO 2 — Verificar Nginx  
sudo systemctl status nginx  

PASO 3 — Probar que la API responde  
curl 127.0.0.1:8000/health  
   
#### Pruebas del servicio en AWS 

IP = 13.60.90.39

Health check — verificar que está viva:    
http://13.60.90.39/health  
   

Página principal:    
http://13.60.90.39/    

Predicción GET — apartamento en Sol:    
http://13.60.90.39/predict-query?neighbourhood_cleansed=Sol&room_type=Entire%20home/apt&accommodates=4&bathrooms=1&bedrooms=2&beds=2&minimum_nights=2&number_of_reviews=50&review_scores_rating=4.5&availability_365=200  
 

Predicción GET — habitación privada en Goya:    
http://13.60.90.39/predict-query?neighbourhood_cleansed=Goya&room_type=Private%20room&accommodates=2&bathrooms=1&bedrooms=1&beds=1&minimum_nights=3&number_of_reviews=30&review_scores_rating=4.2&availability_365=150  


{"predicted_price":87.64}


#### Pruebas del servicio en Render
Mensaje inicio

https://online-ds-bridge-proyects-arielaastorga.onrender.com/

Health  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/health


PATH  
https://online-ds-bridge-proyects-arielaastorga.onrender.com/neighbourhood/Embajadores

Querys:
Piso entero en Sol
https://online-ds-bridge-proyects-arielaastorga.onrender.com/predict-query?accommodates=4&bathrooms=1&bedrooms=2&beds=2&minimum_nights=2&number_of_reviews=50&review_scores_rating=4&availability_365=180&neighbourhood_cleansed=Sol&room_type=Entire%20home/apt

{"predicted_price":183.29}

Habitación privada en Embajadores
https://online-ds-bridge-proyects-arielaastorga.onrender.com/predict-query?accommodates=2&bathrooms=1&bedrooms=1&beds=1&minimum_nights=3&number_of_reviews=120&review_scores_rating=4&availability_365=90&neighbourhood_cleansed=Embajadores&room_type=Private%20room

{"predicted_price":87.64}


In [ ]:
#Ejemplo post en AWS:

import requests

url = "http://13.60.90.39/predict"

payload = {
    "accommodates": 2,
    "bathrooms": 1,
    "bedrooms": 1,
    "beds": 1,
    "minimum_nights": 3,
    "number_of_reviews": 120,
    "review_scores_rating": 4,
    "availability_365": 90,
    "neighbourhood_cleansed": "Embajadores",
    "room_type": "Private room"
}

response = requests.post(url, json=payload)

print(response.status_code)
print(response.json())

200
{'predicccion_precio': 87.64}


# Resumen del flujo

🚪 Puerto 80 → NGINX  
Es el portero del edificio.  

Está escuchando todo lo que llega por internet  
El puerto 80 es la puerta estándar de internet (HTTP)  
Recibe la petición y la redirige internamente a Gunicorn  
No sabe nada de Python ni de tu modelo  

👨‍🍳 Puerto 8000 → GUNICORN  
Es el jefe de sala del restaurante.  

Recibe la petición de Nginx  
Gestiona cuántos trabajadores (workers) atienden peticiones a la vez  
Si llegan 10 peticiones simultáneas, las reparte  
"Traduce" la petición al lenguaje que entiende Flask (protocolo WSGI)  


🍳 Puerto 5000 → FLASK (tu app.py)  
Es el cocinero, tu código Python.  

Recibe los datos del apartamento  
Los pasa al modelo model.pkl  
El modelo calcula el precio predicho  
Devuelve la respuesta en JSON  


📦 Respuesta JSON  
json{"predicted_price": 125.50}  
Viaja de vuelta por el mismo camino hasta el navegador o la app que hizo la petición.  

En resumen con una analogía  
NGINX     = Recepcionista de hotel  
GUNICORN  = Jefe de cocina  
FLASK     = El cocinero (tu app.py)  
JSON      = El plato servido al cliente  
¿Quieres que te explique alguno de los componentes con más detalle?  